# Notebook 04 — CNN Sky Classifier
**Syllabus Week 11 | CLO 4: Convolutional Networks**

## Problem Statement
Train a lightweight ResNet-style CNN to classify sky images into 3 categories:
**sunny**, **cloudy**, **rainy**.

The model is used by the `classify_sky_image` agent tool (via the CLIP zero-shot
wrapper in Notebook 07 which maps CLIP's 6 labels down to these 3).

**Constraint:** < 2 M parameters, CPU training < 10 min.

| Hyperparameter | Value |
|---|---|
| Input | (B, 3, 224, 224) — RGB |
| Output | (B, 3) — logits |
| Classes | sunny · cloudy · rainy |
| Architecture | Stem + 5 ResBlocks |
| Channels | 3→16→32→64→128→128→128 |
| Params | < 2 M |
| Loss | CrossEntropyLoss |
| Optimiser | Adam lr=1e-3 |

## Section 2 — Math Derivation

### Residual Block
Given input feature map $x \in \mathbb{R}^{C \times H \times W}$:

$$\mathcal{F}(x) = W_2 * \text{BN}(\text{ReLU}(W_1 * \text{BN}(x)))$$
$$y = \mathcal{F}(x) + x \quad \text{(skip connection)}$$

When the channel dimension changes, the skip connection uses a 1×1 conv to match:
$$y = \mathcal{F}(x) + W_s * x, \quad W_s \in \mathbb{R}^{C_{out} \times C_{in} \times 1 \times 1}$$

### Cross-Entropy Loss
For a batch of $N$ samples with true labels $y_i \in \{0,1,2\}$ and logits $z_i \in \mathbb{R}^3$:

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log \frac{e^{z_{i,y_i}}}{\sum_{c=0}^{2} e^{z_{i,c}}}$$

### Why Residual Connections?
Without skip connections, gradients must flow through every non-linearity and
normalisation layer. The residual path $y = \mathcal{F}(x) + x$ creates a
gradient superhighway: $\partial \mathcal{L}/\partial x = \partial \mathcal{L}/\partial y \cdot (1 + \partial \mathcal{F}/\partial x)$.
The additive `1` ensures gradients are at least as large as the upstream signal,
preventing vanishing gradients in deeper networks.

### Batch Normalisation
For feature $x$ over a mini-batch $\mathcal{B}$ of size $m$:
$$\hat{x} = \frac{x - \mu_{\mathcal{B}}}{\sqrt{\sigma^2_{\mathcal{B}} + \epsilon}}, \quad
  y = \gamma \hat{x} + \beta$$

At inference, running statistics (not batch statistics) are used — hence
`model.eval()` is required before prediction.

### Metrics
$$\text{Accuracy} = \frac{\text{TP} + \text{TN}}{N}, \quad
  \text{Precision}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FP}_c}, \quad
  \text{Recall}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FN}_c}$$

In [ ]:
# Section 3 — Data Loading
# TODO: provide real sky images in ml/datasets/sky_images/{sunny,cloudy,rainy}/
# Training uses a synthetic fallback (colour-gradient images) when no real images exist.
import sys; sys.path.insert(0, '..')

# from ml.train.train_cnn import SkyImageDataset
# from torchvision import transforms
# tf = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
# ])
# train_ds = SkyImageDataset('train', transform=tf)
# val_ds   = SkyImageDataset('val',   transform=tf)
# test_ds  = SkyImageDataset('test',  transform=tf)
# print(f'train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}')
# img, label = train_ds[0]
# print(f'img shape: {img.shape}')    # (3, 224, 224)
# print(f'label: {label}')            # 0=sunny, 1=cloudy, 2=rainy
# print(f'classes: {train_ds.classes}')

In [ ]:
# Section 4 — Model Definition & Shape Trace
import torch

# from ml.train.train_cnn import SkyCNN
# model = SkyCNN(num_classes=3)
# n = sum(p.numel() for p in model.parameters())
# print(f'Params: {n:,}')        # must be < 2,000,000
# assert n < 2_000_000, f'Model too large: {n:,} params'

# Shape trace:
# x           : (B, 3, 224, 224)     input
# stem (7x7)  : (B, 16, 112, 112)    Conv2d(3,16,7,stride=2) + BN + ReLU
# MaxPool     : (B, 16, 56, 56)
# ResBlock 1  : (B, 16, 56, 56)      in=out=16, no downsample
# ResBlock 2  : (B, 32, 28, 28)      stride=2, skip 1x1
# ResBlock 3  : (B, 64, 14, 14)      stride=2, skip 1x1
# ResBlock 4  : (B, 128, 7, 7)       stride=2, skip 1x1
# ResBlock 5  : (B, 128, 7, 7)       in=out=128, no downsample
# AvgPool     : (B, 128, 1, 1)       AdaptiveAvgPool2d(1)
# flatten     : (B, 128)
# Linear      : (B, 3)               head

# x = torch.randn(2, 3, 224, 224)
# out = model(x)
# print(f'output: {out.shape}')   # (2, 3)

In [ ]:
# Section 5 — Training Results
# Run: python -m ml.train.train_cnn
# Then paste epoch log below:

# TODO: paste training output here
# epoch 01  train_loss=X.XXXX  val_acc=XX.X%
# ...
# Early stop / final epoch
# Test Acc=XX.X%  Prec macro=X.XX  Recall macro=X.XX

from IPython.display import Image
# Image('../models/cnn/train_curve.png')

## Section 6 — Architecture Diagram

```
Input (B, 3, 224, 224)
      │
 Conv2d(3→16, 7×7, stride=2) + BN + ReLU   (B, 16, 112, 112)
      │
 MaxPool2d(3, stride=2)                     (B, 16, 56, 56)
      │
 ┌────┴────────────────────────────────┐
 │  ResBlock 1   16→16  (no stride)   │    (B, 16, 56, 56)
 │  ResBlock 2   16→32  (stride=2)    │    (B, 32, 28, 28)
 │  ResBlock 3   32→64  (stride=2)    │    (B, 64, 14, 14)
 │  ResBlock 4   64→128 (stride=2)    │    (B, 128, 7, 7)
 │  ResBlock 5  128→128 (no stride)   │    (B, 128, 7, 7)
 └────────────────────────────────────┘
      │
 AdaptiveAvgPool2d(1)                       (B, 128, 1, 1)
      │  flatten
 Linear(128 → 3)                            (B, 3)
      │
 Output — logits [sunny | cloudy | rainy]
```

**ResBlock internals:**
```
x ──► Conv(3×3)+BN+ReLU ──► Conv(3×3)+BN ──► + ──► ReLU ──► y
 └─────────────────────── (1×1 conv if C changes) ───┘
```

In [ ]:
# Section 7 — Evaluation: Confusion Matrix + Loss Curve
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image

# 7a. Training curve (loss + val accuracy dual-axis)
# Image('../models/cnn/train_curve.png')

# 7b. Confusion matrix
# Image('../models/cnn/confusion_matrix.png')

# 7c. Per-class precision / recall (paste from training output)
# class       precision  recall
# sunny       TODO       TODO
# cloudy      TODO       TODO
# rainy       TODO       TODO

# 7d. Visualise sample predictions
# import torch
# from torchvision import transforms
# from PIL import Image as PILImage
# model.load_state_dict(torch.load('../models/cnn/model.pt', map_location='cpu'))
# model.eval()
# CLASSES = ['sunny', 'cloudy', 'rainy']
# fig, axes = plt.subplots(1, 5, figsize=(20, 4))
# for i, ax in enumerate(axes):
#     img, label = test_ds[i]
#     with torch.no_grad():
#         logits = model(img.unsqueeze(0))
#         pred = logits.argmax(1).item()
#     ax.imshow(img.permute(1,2,0).numpy() * 0.225 + 0.485)  # rough de-normalize
#     ax.set_title(f'pred={CLASSES[pred]}\ntrue={CLASSES[label]}')
#     ax.axis('off')
# plt.suptitle('CNN Sky Classifier — Sample Predictions')
# plt.tight_layout(); plt.show()

## Section 8 — Comparison vs Baseline & Reflection

### Results (fill in after training)
| Metric | Value |
|---|---|
| Params | TODO (target < 2M) |
| Test accuracy | TODO % |
| Macro precision | TODO |
| Macro recall | TODO |
| Train time | TODO min |

### Design Decisions
| Decision | Alternative | Reason |
|---|---|---|
| ResNet-style blocks | Plain conv stack | Skip connections prevent gradient vanishing at 5+ layers |
| AdaptiveAvgPool(1) | Flatten | Input-size agnostic — any resolution works at inference |
| BN before ReLU | BN after | Standard post-2015 convention; stabilises channel stats |
| Synthetic fallback | Crash if no images | Lets grader verify model shape and training loop without a dataset |
| < 2M params | Deeper/wider | CPU constraint — must train in < 10 min |

### Trade-offs
| Aspect | SkyCNN | Transfer (e.g. MobileNetV2) |
|---|---|---|
| Parameter efficiency | Lower (trained from scratch) | Higher (ImageNet weights) |
| Training data required | More images needed | Can fine-tune on < 100 |
| Interpretability | Filters interpretable | Black-box pre-trained features |
| CPU training time | Fast (< 2M params) | Slow (MobileNetV2 = 3.4M) |

### Limitations
- Synthetic training data (colour gradients) will under-perform on real sky photos.
- 3-class taxonomy collapses nuance (e.g. light overcast vs heavy storm).
- No data augmentation implemented — real deployment needs horizontal flip, colour jitter.

### Numbers to fill in
- Best val accuracy: **TODO**
- Epoch of early stop: **TODO**
- Hardest-to-classify class: **TODO** (expect cloudy — most ambiguous)